In [ ]:
import pandas as pd

In [ ]:
df_index = pd.read_excel(r'shared_data_read_only/District_and_School_Performance_Index_Ranking.xlsx', sheet_name='Building PI Rankings')
df_value = pd.read_excel(r'shared_data_read_only/District_Value_Added_Ranking.xlsx', sheet_name='Value-Added Rankings 2023')
#df_lookup = pd.read_csv(r'shared_data_read_only/NCES_School_Lookup.csv')

In [ ]:
df_merge = df_index.merge(df_value, left_on='LEA IRN', right_on='District IRN')
df_merge = df_merge[df_merge['2023 PI for Ranking'] != 'NC']

In [ ]:
df_new = df_merge.groupby('ODE Designated County_x').size().reset_index(name='counts').sort_values(by='counts')
df_new = df_new[df_new['counts']>20]

In [ ]:
counties = []
for i in range(len(df_new)):
    counties.append(df_new.iloc[i,0])

In [ ]:
df_merge['2023 PI for Ranking'] = df_merge["2023 PI for Ranking"].astype(float)
grouped_county = df_merge.groupby('ODE Designated County_x')
grouped_dict_variance_county = grouped_county['2023 PI for Ranking'].var().to_dict()
grouped_dict_counts_county = grouped_county.size().to_dict()

In [ ]:
df_merge['Variance'] = None
for index, row in df_merge.iterrows():
    currCounty = row['ODE Designated County_x']
    if grouped_dict_counts_county[currCounty] >= 20:
        df_merge.at[index, 'Variance'] = grouped_dict_variance_county[currCounty]
    else:
        df_merge.at[index, 'Variance'] = 1000

In [ ]:
df_merge.to_excel('variance.xlsx', columns = ['ODE Designated County_x', 'Variance'])